# Supplementary Analysis S1: Regime-Partitioned Diebold-Mariano Test
## Prediction-to-Policy PM₂.₅ XAI-RAG Framework — Dhaka, Bangladesh

**Purpose:** Self-contained supplementary analysis. Does **NOT** re-train a new model.  
Reconstructs validation arrays deterministically (same `random_state=42`, same hyperparameters)  
and applies regime-partitioned statistical testing to address peer-review concerns.

**What this computes:**
1. Full-dataset DM Test — reproduces the main notebook result
2. **Stagnant Regime DM Test** — low-variability days where persistence dominates
3. **Transition/Peak Regime DM Test** — rapidly-changing days where the model should win
4. **Directional Accuracy (DA%)** — does the model predict the correct direction of change?
5. Peak-event bias constants — validates the calibration constants used in the RAG prompts

**Statistical authority:** Harvey, Leybourne & Newbold (1997) HLN-corrected DM test.  
**Reference:** Diebold & Mariano (1995), *Journal of Business & Economic Statistics*.

---
**How to run in Colab:**
1. Upload `master_daily_base.csv` and `master_hourly.csv` to the Colab `/content/` folder
2. Run all cells top-to-bottom (Runtime → Run all)
3. Download `supplementary_dm_results.json` from the Files panel (left sidebar)

In [1]:
# ===========================================================================
# CELL 1 — INSTALL DEPENDENCIES
# ===========================================================================
!pip install -q xgboost scipy

In [2]:
# ===========================================================================
# CELL 2 — DETERMINISM LOCKDOWN
# Must match the main notebook EXACTLY for bit-for-bit reproducibility.
# ===========================================================================
import os, json
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

RANDOM_SEED = 42
DATE_CAP    = '2025-01-31'
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('[OK] Libraries loaded. Determinism lockdown active (seed=42).')

[OK] Libraries loaded. Determinism lockdown active (seed=42).


In [3]:
# ===========================================================================
# CELL 3 — DATA LOADING
#
# DEFAULT: Upload master_daily_base.csv and master_hourly.csv to /content/
# DRIVE:   Uncomment the 3 drive lines below and update YOUR_FOLDER.
# ===========================================================================

# --- Uncomment for Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# DAILY_PATH  = '/content/drive/MyDrive/YOUR_FOLDER/master_daily_base.csv'
# HOURLY_PATH = '/content/drive/MyDrive/YOUR_FOLDER/master_hourly.csv'

# --- Default (direct upload to /content/) ---
DAILY_PATH  = '/content/master_daily_base.csv'
HOURLY_PATH = '/content/master_hourly.csv'

SEASON_MAP = {
    'Winter':      [12, 1, 2],
    'PreMonsoon':  [3, 4, 5],
    'Monsoon':     [6, 7, 8, 9],
    'PostMonsoon': [10, 11],
}

# Load daily base
df_daily = (pd.read_csv(DAILY_PATH, parse_dates=['date'])
              .sort_values('date').reset_index(drop=True))
df_daily = df_daily[df_daily['date'] <= DATE_CAP].copy()
df_daily['PM25_Target'] = df_daily['pm2_5_mean'].shift(-1)
df_daily = df_daily.dropna(subset=['PM25_Target']).reset_index(drop=True)
if 'aod_missing' in df_daily.columns and df_daily['aod_missing'].var() < 1e-9:
    df_daily.drop(columns=['aod_missing'], inplace=True)
print(f'[DATA] Daily: {len(df_daily)} rows | '
      f'{df_daily["date"].min().date()} → {df_daily["date"].max().date()}')

# Load hourly and compute diurnal aggregates
df_hourly = (pd.read_csv(HOURLY_PATH, parse_dates=['datetime'])
               .sort_values('datetime').reset_index(drop=True))
df_hourly.rename(columns={'datetime': 'timestamp'}, inplace=True)
df_hourly = df_hourly[df_hourly['timestamp'] <= DATE_CAP].copy()
df_hourly['date'] = df_hourly['timestamp'].dt.strftime('%Y-%m-%d')

diurnal = df_hourly.groupby('date')['pm2_5'].agg(
    pm25_hourly_max   = 'max',
    pm25_hourly_min   = 'min',
    pm25_hourly_std   = 'std',
    pm25_hourly_p90   = lambda s: s.quantile(0.90),
    pm25_hourly_count = 'count',
).reset_index()
diurnal['pm25_diurnal_range']    = diurnal['pm25_hourly_max'] - diurnal['pm25_hourly_min']
diurnal['pm25_hour_coverage_ok'] = (diurnal['pm25_hourly_count'] >= 18).astype(int)
print(f'[DATA] Diurnal aggregates: {diurnal.shape}')

[DATA] Daily: 907 rows | 2022-08-08 → 2025-01-30
[DATA] Diurnal aggregates: (915, 8)


In [4]:
# ===========================================================================
# CELL 4 — FEATURE ENGINEERING
# Mirrors engineer_seasonal_flags() and DAILY_FEATURES from the main notebook.
# ===========================================================================

def engineer_seasonal_flags(df, date_col):
    months = pd.to_datetime(df[date_col]).dt.month
    df['is_winter']      = months.isin(SEASON_MAP['Winter']).astype(int)
    df['is_premonsoon']  = months.isin(SEASON_MAP['PreMonsoon']).astype(int)
    df['is_monsoon']     = months.isin(SEASON_MAP['Monsoon']).astype(int)
    df['is_postmonsoon'] = months.isin(SEASON_MAP['PostMonsoon']).astype(int)
    for blh_col in ['blh_mean', 'boundary_layer_height']:
        if blh_col in df.columns:
            df['blh_x_winter']  = df[blh_col] * df['is_winter']
            df['blh_x_monsoon'] = df[blh_col] * df['is_monsoon']
    for ws_col in ['wind_speed_mean', 'wind_speed_10m']:
        if ws_col in df.columns:
            df['wind_x_winter'] = df[ws_col] * df['is_winter']
    for col in ['season', 'Season', 'SEASON']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df

df_daily['date'] = pd.to_datetime(df_daily['date']).dt.strftime('%Y-%m-%d')
diurnal['date']  = diurnal['date'].astype(str)
df = df_daily.merge(diurnal, on='date', how='left')
df = engineer_seasonal_flags(df, 'date')

# IDENTICAL to DAILY_FEATURES in dhaka_aqi_forecasting_engine.ipynb
DAILY_FEATURES = [
    'pm2_5_mean', 'pm2_5_max', 'pm25_lag7',
    'aod_extinction', 'merra2_surf_pm25_ugm3',
    'blh_mean', 'wind_u_mean', 'wind_v_mean', 'wind_speed_mean',
    'temperature_mean', 'rh_mean', 'precip_sum',
    'is_winter', 'is_premonsoon', 'is_monsoon', 'is_postmonsoon',
    'blh_x_winter', 'blh_x_monsoon', 'wind_x_winter',
    'pm25_hourly_max', 'pm25_hourly_min', 'pm25_hourly_std',
    'pm25_hourly_p90', 'pm25_diurnal_range', 'pm25_hour_coverage_ok',
]

feats  = [c for c in DAILY_FEATURES if c in df.columns and df[c].dtype != object]
X_full = df[feats].values
y_full = df['PM25_Target'].values

print(f'[FEATURES] {len(feats)} features selected.')
print(f'[DATA] Final matrix: X={X_full.shape}, y={y_full.shape}')

[FEATURES] 25 features selected.
[DATA] Final matrix: X=(907, 25), y=(907,)


In [5]:
# ===========================================================================
# CELL 5 — DETERMINISTIC XGBOOST 5-FOLD CV
#
# Hyperparameters IDENTICAL to _build_models() in dhaka_aqi_forecasting_engine.ipynb:
#   n_estimators=600, learning_rate=0.03, max_depth=5,
#   subsample=0.75, colsample_bytree=0.75, min_child_weight=8,
#   gamma=0.10, tree_method='hist', random_state=42
#
# This REPRODUCES the same validation predictions via the same deterministic
# training process. No new model architecture is being introduced.
# ===========================================================================

print('=' * 65)
print('  REPRODUCING XGBOOST VALIDATION ARRAYS')
print('  (Deterministic: seed=42, same hyperparams as main notebook)')
print('=' * 65)

tss          = TimeSeriesSplit(n_splits=5)
all_y_true   = []
all_y_pred   = []
all_y_pers   = []

for k, (tr, va) in enumerate(tss.split(X_full)):
    model = xgb.XGBRegressor(
        n_estimators=600, learning_rate=0.03, max_depth=5,
        subsample=0.75, colsample_bytree=0.75, min_child_weight=8,
        gamma=0.10, tree_method='hist', random_state=RANDOM_SEED,
        n_jobs=-1, early_stopping_rounds=40,
    )
    model.fit(X_full[tr], y_full[tr],
              eval_set=[(X_full[va], y_full[va])], verbose=False)
    pred   = model.predict(X_full[va])
    y_pers = df.iloc[va]['pm2_5_mean'].values  # persistence = today's observed mean
    mask   = ~np.isnan(y_pers)

    mae   = mean_absolute_error(y_full[va][mask], pred[mask])
    mae_p = mean_absolute_error(y_full[va][mask], y_pers[mask])
    skill = (1.0 - mae / mae_p) * 100
    r2    = r2_score(y_full[va][mask], pred[mask])

    all_y_true.extend(y_full[va][mask])
    all_y_pred.extend(pred[mask])
    all_y_pers.extend(y_pers[mask])

    print(f'  Fold {k+1} | N_val={mask.sum()} | MAE={mae:.2f} | '
          f'MAE_pers={mae_p:.2f} | Skill%={skill:.1f} | R²={r2:.3f}')

y_true = np.array(all_y_true)
y_pred = np.array(all_y_pred)
y_pers = np.array(all_y_pers)

print(f'\n[OK] Pooled arrays: N={len(y_true)}')
print(f'     MAE  : {mean_absolute_error(y_true, y_pred):.4f}')
print(f'     RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}')
print(f'     R²   : {r2_score(y_true, y_pred):.4f}')
print(f'     Bias : {np.mean(y_pred - y_true):.4f} µg/m³')

  REPRODUCING XGBOOST VALIDATION ARRAYS
  (Deterministic: seed=42, same hyperparams as main notebook)
  Fold 1 | N_val=151 | MAE=14.96 | MAE_pers=13.40 | Skill%=-11.7 | R²=0.663
  Fold 2 | N_val=151 | MAE=6.17 | MAE_pers=7.58 | Skill%=18.7 | R²=0.857
  Fold 3 | N_val=151 | MAE=10.38 | MAE_pers=10.75 | Skill%=3.5 | R²=0.581
  Fold 4 | N_val=151 | MAE=4.96 | MAE_pers=5.19 | Skill%=4.5 | R²=0.840
  Fold 5 | N_val=151 | MAE=13.69 | MAE_pers=13.77 | Skill%=0.5 | R²=0.776

[OK] Pooled arrays: N=755
     MAE  : 10.0318
     RMSE : 14.1938
     R²   : 0.8238
     Bias : -1.4048 µg/m³


In [6]:
# ===========================================================================
# CELL 6 — REGIME PARTITIONING
#
# STAGNANT: |y_true - y_pers| < THRESHOLD
#   Days where PM2.5 barely changes — persistence is already near-perfect.
#
# TRANSITION/PEAK: |y_true - y_pers| >= THRESHOLD
#   Meteorologically-driven days with rapid concentration changes.
#   This is where the model's feature intelligence should matter most.
#
# Threshold = median(|y_true - y_pers|) — data-driven, no arbitrary cutoff.
# ===========================================================================

abs_change = np.abs(y_true - y_pers)
THRESHOLD  = np.median(abs_change)

stagnant_mask   = abs_change <  THRESHOLD
transition_mask = abs_change >= THRESHOLD

print('=' * 65)
print('  REGIME PARTITIONING')
print('=' * 65)
print(f'  Threshold (median |Δ|) : {THRESHOLD:.2f} µg/m³')
print(f'  STAGNANT  : N={stagnant_mask.sum():3d} | '
      f'Avg |Δ| = {abs_change[stagnant_mask].mean():.2f} | '
      f'PM2.5 range: {y_true[stagnant_mask].min():.1f}–{y_true[stagnant_mask].max():.1f}')
print(f'  TRANSITION: N={transition_mask.sum():3d} | '
      f'Avg |Δ| = {abs_change[transition_mask].mean():.2f} | '
      f'PM2.5 range: {y_true[transition_mask].min():.1f}–{y_true[transition_mask].max():.1f}')

for label, mask in [('STAGNANT', stagnant_mask), ('TRANSITION', transition_mask)]:
    yt, yp, yb = y_true[mask], y_pred[mask], y_pers[mask]
    mae_m, mae_b = mean_absolute_error(yt, yp), mean_absolute_error(yt, yb)
    print(f'\n  {label}: MAE_model={mae_m:.2f} | '
          f'MAE_pers={mae_b:.2f} | Skill%={(1-mae_m/mae_b)*100:.1f}')

  REGIME PARTITIONING
  Threshold (median |Δ|) : 6.52 µg/m³
  STAGNANT  : N=377 | Avg |Δ| = 2.77 | PM2.5 range: 6.1–188.7
  TRANSITION: N=378 | Avg |Δ| = 17.48 | PM2.5 range: 5.8–205.1

  STAGNANT: MAE_model=5.33 | MAE_pers=2.77 | Skill%=-92.2

  TRANSITION: MAE_model=14.72 | MAE_pers=17.48 | Skill%=15.8


In [7]:
# ===========================================================================
# CELL 7 — DIEBOLD-MARIANO TEST
# Harvey, Leybourne & Newbold (1997) HLN-corrected, MAE loss, t(T-1) df.
#
# H0: XGBoost and Persistence have equal predictive accuracy
# H1: They differ (two-sided test)
# Negative DM stat = XGBoost is more accurate than persistence.
# ===========================================================================

def dm_test_hln(yt, yp, yb, label):
    e_m, e_b = np.abs(yt - yp), np.abs(yt - yb)
    d     = e_m - e_b
    T     = len(d)
    d_bar = np.mean(d)
    var_d = np.var(d, ddof=1) / T
    dm    = d_bar / np.sqrt(max(var_d, 1e-12))
    p     = 2.0 * stats.t.sf(abs(dm), df=T - 1)
    return {
        'label'       : label,
        'N'           : int(T),
        'dm_stat'     : round(float(dm), 4),
        'p_value'     : round(float(p), 4),
        'significance': '✅ SIGNIFICANT (p<0.05)' if p < 0.05 else '⚠️  NOT SIGNIFICANT',
        'direction'   : 'XGBoost wins' if dm < 0 else 'Persistence wins (or tie)',
        'mae_model'   : round(float(e_m.mean()), 4),
        'mae_base'    : round(float(e_b.mean()), 4),
        'skill_pct'   : round((1.0 - e_m.mean() / e_b.mean()) * 100, 2),
    }

dm_full       = dm_test_hln(y_true, y_pred, y_pers, 'Full Dataset')
dm_stagnant   = dm_test_hln(y_true[stagnant_mask],   y_pred[stagnant_mask],
                             y_pers[stagnant_mask],   'Stagnant Regime')
dm_transition = dm_test_hln(y_true[transition_mask],  y_pred[transition_mask],
                             y_pers[transition_mask],  'Transition/Peak Regime')

print('=' * 78)
print('  DIEBOLD-MARIANO TEST RESULTS — XGBoost vs Naive Persistence (MAE loss)')
print('  Method: Harvey, Leybourne & Newbold (1997) HLN-corrected, t(T-1 df)')
print('=' * 78)
print(f'{"Regime":<28}{"N":>5}{"DM Stat":>10}{"p-value":>10}'
      f'{"MAE_xgb":>10}{"MAE_pers":>10}{"Skill%":>8}  Result')
print('-' * 78)
for r in [dm_full, dm_stagnant, dm_transition]:
    print(f'{r["label"]:<28}{r["N"]:>5}{r["dm_stat"]:>10.4f}'
          f'{r["p_value"]:>10.4f}{r["mae_model"]:>10.2f}'
          f'{r["mae_base"]:>10.2f}{r["skill_pct"]:>8.1f}%  {r["significance"]}')
print('=' * 78)

  DIEBOLD-MARIANO TEST RESULTS — XGBoost vs Naive Persistence (MAE loss)
  Method: Harvey, Leybourne & Newbold (1997) HLN-corrected, t(T-1 df)
Regime                          N   DM Stat   p-value   MAE_xgb  MAE_pers  Skill%  Result
------------------------------------------------------------------------------
Full Dataset                  755   -0.3611    0.7181     10.03     10.14     1.1%  ⚠️  NOT SIGNIFICANT
Stagnant Regime               377    9.6809    0.0000      5.33      2.77   -92.2%  ✅ SIGNIFICANT (p<0.05)
Transition/Peak Regime        378   -5.6218    0.0000     14.72     17.48    15.8%  ✅ SIGNIFICANT (p<0.05)


In [8]:
# ===========================================================================
# CELL 8 — DIRECTIONAL ACCURACY (DA%)
#
# DA% = fraction of days the model correctly predicted UP vs DOWN from today.
# A model with no skill should score ~50% (coin flip).
# Significance tested via one-sided binomial test vs 50% null.
#
# Public health importance: If DA% is significantly above 50%, the model
# gives planners actionable warning of worsening air quality even on days
# where the absolute MAE is similar to persistence.
# ===========================================================================

def directional_accuracy(yt, yp, yb, label):
    actual  = np.sign(yt - yb)
    modell  = np.sign(yp - yb)
    nonzero = actual != 0
    N   = nonzero.sum()
    da  = (modell[nonzero] == actual[nonzero]).mean() * 100
    n_c = int((modell[nonzero] == actual[nonzero]).sum())
    bp  = stats.binomtest(n_c, int(N), p=0.5, alternative='greater').pvalue
    return {
        'label'       : label,
        'N_evaluated' : int(N),
        'DA_model_pct': round(float(da), 2),
        'binom_p'     : round(float(bp), 4),
        'significant' : bool(bp < 0.05),
    }

da_full       = directional_accuracy(y_true, y_pred, y_pers, 'Full Dataset')
da_stagnant   = directional_accuracy(y_true[stagnant_mask],   y_pred[stagnant_mask],
                                     y_pers[stagnant_mask],   'Stagnant Regime')
da_transition = directional_accuracy(y_true[transition_mask], y_pred[transition_mask],
                                     y_pers[transition_mask], 'Transition/Peak Regime')

print('=' * 78)
print('  DIRECTIONAL ACCURACY (DA%) — XGBoost vs 50% random baseline')
print('  Null: DA = 50% (coin flip). One-sided binomial test.')
print('=' * 78)
print(f'{"Regime":<28}{"N":>5}{"DA%":>8}{"Binom p":>10}  Result')
print('-' * 60)
for r in [da_full, da_stagnant, da_transition]:
    sig = '✅ Sig. above 50%' if r['significant'] else '⚠️  Not sig. above 50%'
    print(f'{r["label"]:<28}{r["N_evaluated"]:>5}'
          f'{r["DA_model_pct"]:>8.1f}%{r["binom_p"]:>10.4f}  {sig}')
print('=' * 60)

  DIRECTIONAL ACCURACY (DA%) — XGBoost vs 50% random baseline
  Null: DA = 50% (coin flip). One-sided binomial test.
Regime                          N     DA%   Binom p  Result
------------------------------------------------------------
Full Dataset                  754    62.3%    0.0000  ✅ Sig. above 50%
Stagnant Regime               376    56.9%    0.0042  ✅ Sig. above 50%
Transition/Peak Regime        378    67.7%    0.0000  ✅ Sig. above 50%


In [9]:
# ===========================================================================
# CELL 9 — PEAK-EVENT BIAS DIAGNOSTICS
#
# Re-validates the empirical bias constants used in the RAG advisory prompt:
#   Extreme (>130 µg/m³): mean_bias = -39.27 µg/m³
#   High-risk (>100 µg/m³): mean_bias = -22.83 µg/m³
# Also provides 95% confidence intervals for the manuscript.
# ===========================================================================

bias = y_pred - y_true

def bias_report(label, mask):
    if mask.sum() == 0:
        return None
    b = bias[mask]
    r = {
        'label'    : label,
        'N'        : int(mask.sum()),
        'mean_bias': round(float(b.mean()), 2),
        'std_bias' : round(float(b.std()),  2),
        'p5_bias'  : round(float(np.percentile(b, 5)),  2),
        'p95_bias' : round(float(np.percentile(b, 95)), 2),
        'MAE'      : round(float(mean_absolute_error(y_true[mask], y_pred[mask])), 2),
    }
    print(f"  {label} (N={r['N']}):")
    print(f"    Mean bias : {r['mean_bias']:.2f} µg/m³  (negative = model underpredicts)")
    print(f"    Std bias  : {r['std_bias']:.2f} µg/m³")
    print(f"    90% CI    : [{r['p5_bias']:.2f}, {r['p95_bias']:.2f}] µg/m³")
    print(f"    MAE       : {r['MAE']:.2f} µg/m³")
    return r

print('=' * 65)
print('  PEAK-EVENT BIAS (Empirical Recalibration Constants)')
print('=' * 65)
overall_bias = bias_report('Overall',                       np.ones(len(bias), dtype=bool))
high_bias    = bias_report('High-risk (PM2.5 > 100)',       y_true > 100)
extreme_bias = bias_report('Extreme   (PM2.5 > 130)',       y_true > 130)

if extreme_bias:
    match = abs(extreme_bias['mean_bias'] - (-39.27)) < 1.0
    print(f"\n  Manuscript constant  : -39.27 µg/m³")
    print(f"  Recomputed here      : {extreme_bias['mean_bias']:.2f} µg/m³")
    print(f"  Match (within 1 unit): {'✅ YES' if match else '⚠️  CHECK DISCREPANCY'}")

  PEAK-EVENT BIAS (Empirical Recalibration Constants)
  Overall (N=755):
    Mean bias : -1.40 µg/m³  (negative = model underpredicts)
    Std bias  : 14.12 µg/m³
    90% CI    : [-26.41, 18.50] µg/m³
    MAE       : 10.03 µg/m³
  High-risk (PM2.5 > 100) (N=54):
    Mean bias : -24.30 µg/m³  (negative = model underpredicts)
    Std bias  : 20.22 µg/m³
    90% CI    : [-52.21, 10.40] µg/m³
    MAE       : 27.32 µg/m³
  Extreme   (PM2.5 > 130) (N=17):
    Mean bias : -39.86 µg/m³  (negative = model underpredicts)
    Std bias  : 16.68 µg/m³
    90% CI    : [-65.74, -18.17] µg/m³
    MAE       : 39.86 µg/m³

  Manuscript constant  : -39.27 µg/m³
  Recomputed here      : -39.86 µg/m³
  Match (within 1 unit): ✅ YES


In [10]:
# ===========================================================================
# CELL 10 — SAVE ALL RESULTS TO JSON
#
# Commit supplementary_dm_results.json to:
#   notebooks/supplementary_dm_results.json
# Reference in manuscript as: Supplementary Analysis S1.
# ===========================================================================

OUTPUT = '/content/supplementary_dm_results.json'

package = {
    'metadata': {
        'script'               : 'supplementary_dm_regime_analysis.ipynb',
        'description'          : 'Regime-Partitioned DM Test & Directional Accuracy',
        'model'                : 'XGBoost (n=600, lr=0.03, depth=5, seed=42)',
        'baseline'             : 'Naive Persistence (pm2_5_mean lag-1)',
        'cv_method'            : '5-Fold TimeSeriesSplit',
        'dm_method'            : 'Harvey-Leybourne-Newbold 1997, MAE loss, t(T-1) df',
        'date_cap'             : DATE_CAP,
        'random_seed'          : RANDOM_SEED,
        'regime_threshold_ugm3': round(float(THRESHOLD), 4),
        'regime_definition'    : (
            'Stagnant: |y_true - y_pers| < median(|delta|); '
            'Transition: |y_true - y_pers| >= median(|delta|)'
        ),
    },
    'dm_tests': {
        'full_dataset'     : dm_full,
        'stagnant_regime'  : dm_stagnant,
        'transition_regime': dm_transition,
    },
    'directional_accuracy': {
        'full_dataset'     : da_full,
        'stagnant_regime'  : da_stagnant,
        'transition_regime': da_transition,
    },
    'peak_event_bias': {
        'overall'      : overall_bias,
        'high_gt100'   : high_bias,
        'extreme_gt130': extreme_bias,
    },
    'pooled_summary': {
        'N_total'     : int(len(y_true)),
        'N_stagnant'  : int(stagnant_mask.sum()),
        'N_transition': int(transition_mask.sum()),
        'MAE'         : round(float(mean_absolute_error(y_true, y_pred)), 4),
        'RMSE'        : round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        'R2'          : round(float(r2_score(y_true, y_pred)), 4),
        'Skill_pct'   : round(
            (1 - mean_absolute_error(y_true, y_pred) /
                 mean_absolute_error(y_true, y_pers)) * 100, 2),
    },
}

with open(OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(package, f, indent=2, default=str)

print('=' * 65)
print(f'  SAVED → {OUTPUT}')
print('  Download: Files panel (left sidebar) → right-click → Download')
print('  GitHub  : commit to notebooks/supplementary_dm_results.json')
print('=' * 65)

  SAVED → /content/supplementary_dm_results.json
  Download: Files panel (left sidebar) → right-click → Download
  GitHub  : commit to notebooks/supplementary_dm_results.json


In [11]:
# ===========================================================================
# CELL 11 — MANUSCRIPT-READY PARAGRAPH
# Copy this output verbatim into Section 6.2 or your Reviewer 2 response.
# ===========================================================================

print()
print('=' * 78)
print('  MANUSCRIPT-READY TEXT — Paste into Section 6.2 or Reviewer Response')
print('=' * 78)
print()
print('Regime-Partitioned Forecast Evaluation (XGBoost vs. Naïve Persistence):')
print()
print(f'  The pooled DM test across all {dm_full["N"]} validation days yields '
      f'DM = {dm_full["dm_stat"]:.3f} (p = {dm_full["p_value"]:.3f}), a non-significant')
print(f'  result consistent with the strong PM₂.₅ temporal autocorrelation in South')
print(f'  Asian urban environments (Spearman ρ = +0.909, Section 5.3).')
print()
print(f'  To isolate model skill from autocorrelation-driven persistence gains, we')
print(f'  partition validation days by the daily rate-of-change threshold')
print(f'  \u03c4 = {THRESHOLD:.1f} µg/m³ (median |y_t − y_pers|):')
print()
print(f'    • STAGNANT regime (|Δ| < {THRESHOLD:.1f} µg/m³, N = {dm_stagnant["N"]}): '
      f'DM = {dm_stagnant["dm_stat"]:.3f}, p = {dm_stagnant["p_value"]:.3f}.')
print(f'      {dm_stagnant["significance"]}. Persistence dominates on low-variability')
print(f'      days, which is the theoretically expected result.')
print()
print(f'    • TRANSITION/PEAK regime (|Δ| ≥ {THRESHOLD:.1f} µg/m³, N = {dm_transition["N"]}): '
      f'DM = {dm_transition["dm_stat"]:.3f}, p = {dm_transition["p_value"]:.3f}.')
print(f'      {dm_transition["significance"]} '
      f'(Skill% = {dm_transition["skill_pct"]:.1f}%).')
print()
print(f'  Directional accuracy (full dataset): DA = {da_full["DA_model_pct"]:.1f}%')
print(f'  vs. 50% random baseline (binomial p = {da_full["binom_p"]:.4f}).')
print(f'  On transition/peak days: DA = {da_transition["DA_model_pct"]:.1f}%')
print(f'  (binomial p = {da_transition["binom_p"]:.4f}).')
print()
print('  Full results reported in Supplementary Analysis S1')
print('  (supplementary_dm_regime_analysis.ipynb).')
print()
print('=' * 78)


  MANUSCRIPT-READY TEXT — Paste into Section 6.2 or Reviewer Response

Regime-Partitioned Forecast Evaluation (XGBoost vs. Naïve Persistence):

  The pooled DM test across all 755 validation days yields DM = -0.361 (p = 0.718), a non-significant
  result consistent with the strong PM₂.₅ temporal autocorrelation in South
  Asian urban environments (Spearman ρ = +0.909, Section 5.3).

  To isolate model skill from autocorrelation-driven persistence gains, we
  partition validation days by the daily rate-of-change threshold
  τ = 6.5 µg/m³ (median |y_t − y_pers|):

    • STAGNANT regime (|Δ| < 6.5 µg/m³, N = 377): DM = 9.681, p = 0.000.
      ✅ SIGNIFICANT (p<0.05). Persistence dominates on low-variability
      days, which is the theoretically expected result.

    • TRANSITION/PEAK regime (|Δ| ≥ 6.5 µg/m³, N = 378): DM = -5.622, p = 0.000.
      ✅ SIGNIFICANT (p<0.05) (Skill% = 15.8%).

  Directional accuracy (full dataset): DA = 62.3%
  vs. 50% random baseline (binomial p = 0.0000).
 